##### CNN plot

In [ ]:
import visualkeras
from PIL import ImageFont
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Flatten, Dense, InputLayer

if __name__ == "__main__":
    model = Sequential([
        InputLayer(input_shape=(18, 29, 29)),
        Conv2D(64, kernel_size=3, padding='same', data_format='channels_last'),
        BatchNormalization(axis=1),
        MaxPooling2D(pool_size=(2, 2), data_format='channels_last'),
        Conv2D(128, kernel_size=3, padding='same', data_format='channels_last'),
        BatchNormalization(axis=1),
        MaxPooling2D(pool_size=(2, 2), data_format='channels_last'),
        Conv2D(256, kernel_size=3, padding='same', data_format='channels_last'),
        BatchNormalization(axis=1),
        MaxPooling2D(pool_size=(2, 2), data_format='channels_last'),
        Conv2D(512, kernel_size=3, padding='same', data_format='channels_last'),
        BatchNormalization(axis=1),
        MaxPooling2D(pool_size=(2, 2), data_format='channels_last'),
        Flatten(),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(2, activation=None),
    ])

    # model.build(input_shape=(None, 18, 29, 29))

    
    font = ImageFont.truetype("../times-new-roman/times.ttf", size=40)
    #model.add(visualkeras.SpacingDummyLayer(spacing=100))

    img = visualkeras.layered_view(
        model,
        legend=True,                             # show legend
        draw_volume=True,                        # show volume of each layer
        # show_dimension=True,                   # show dimension of each layer
        font=font,
        spacing=35,                              # spacing between layers
        scale_xy=30,                            
        scale_z=0.5,                             
    )
    # img.save("../result_DL/model_plot/CNN_visualization.png", format="PNG")  
    img.show()  




c:\Users\NESS-Kuan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
c:\Users\NESS-Kuan\AppData\Local\Programs\Python\Python311\Lib\site-packages\visualkeras\layered.py:86: UserWarning: The legend_text_spacing_offset parameter is deprecated and will be removed in a future release.
  warnings.warn("The legend_text_spacing_offset parameter is deprecated and will be removed in a future release.")


##### 2D VAE plot

In [223]:
import math
import visualkeras
from PIL import ImageFont
import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    Conv2DTranspose,
    LeakyReLU,
    Flatten,
    Dense,
    Concatenate,
    Reshape
)
from tensorflow.keras.models import Model





in_ch       = 18    
latent_dim  = 50    
num_classes = 2     
H, W        = 29, 29


ds_h = math.ceil(H / 8)  # ceil(29/8) = 4
ds_w = math.ceil(W / 8)  # = 4
flattened_size = 256 * ds_h * ds_w  # 256 * 4 * 4 = 4096



# color design
color_map = {
    tf.keras.layers.Conv2D:          {"fill": "orange"},
    tf.keras.layers.Conv2DTranspose: {"fill": "#ffd166"},
    tf.keras.layers.LeakyReLU:       {"fill": "#ef476f"},
    tf.keras.layers.Dense:           {"fill": "#118ab2"},
    tf.keras.layers.Flatten:         {"fill": "#842da1"},
    tf.keras.layers.Concatenate:     {"fill": "#ffbad4"},
    tf.keras.layers.Reshape:         {"fill": "lightblue"},
    InputLayer:                      {"fill": "#fe9775"},

}



# encoder
x_input = Input(shape=(H, W, in_ch), name="x_input")
c_input = Input(shape=(num_classes,), name="c_input")

h = Conv2D(64, kernel_size=3, strides=2, padding="same", name="enc_conv1")(x_input)
h = LeakyReLU(alpha=0.2, name="enc_lrelu1")(h)

h = Conv2D(128, kernel_size=3, strides=2, padding="same", name="enc_conv2")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu2")(h)

h = Conv2D(256, kernel_size=3, strides=2, padding="same", name="enc_conv3")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu3")(h)
# h.shape = (None, 4, 4, 256)

h = Flatten(name="enc_flatten")(h)  # → (None, 4096)
h = Concatenate(name="enc_concat")([h, c_input])  # → (None, 4096 + num_classes)

h = Dense(512, name="enc_fc1")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu4")(h)
h = Dense(256, name="enc_fc2")(h)
h = LeakyReLU(alpha=0.2, name="enc_lrelu5")(h)

mu     = Dense(latent_dim, name="enc_mu")(h)
logvar = Dense(latent_dim, name="enc_logvar")(h)

encoder = Model(inputs=[x_input, c_input], outputs=[mu, logvar], name="Encoder")




# decoder
z_input   = Input(shape=(latent_dim,),  name="z_input")
c_input_d = Input(shape=(num_classes,), name="c_input_d")

zd = Concatenate(name="dec_concat")([z_input, c_input_d])  # → (None, latent_dim + num_classes)

d = Dense(256, name="dec_fc1")(zd)
d = LeakyReLU(alpha=0.2, name="dec_lrelu1")(d)
d = Dense(512, name="dec_fc2")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu2")(d)
d = Dense(flattened_size, name="dec_fc3")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu3")(d)

d = Reshape((ds_h, ds_w, 256), name="dec_reshape")(d)
# d.shape = (None, 4, 4, 256)

d = Conv2DTranspose(64, kernel_size=3, strides=2, padding="same", name="dec_deconv1")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu4")(d)
# d.shape = (None, 8, 8, 64)

d = Conv2DTranspose(32, kernel_size=3, strides=2, padding="same", name="dec_deconv2")(d)
d = LeakyReLU(alpha=0.2, name="dec_lrelu5")(d)
# d.shape = (None, 16, 16, 32)

d = Conv2DTranspose(in_ch, kernel_size=3, strides=2, padding="same", name="dec_deconv3")(d)
# d.shape = (None, 32, 32, in_ch)

d = Cropping2D(cropping=((0, 3), (0, 3)), name="dec_crop")(d)
decoder = Model(inputs=[z_input, c_input_d], outputs=d, name="Decoder")




# full model
full_output = decoder([mu, c_input])
full_model  = Model(inputs=[x_input, c_input], outputs=full_output, name="cVAE2d_Full")




if __name__ == "__main__":
    try:
        font = ImageFont.truetype("../times-new-roman/times.ttf", size=40)
    except:
        font = None

    # plot en
    encoder.build(input_shape=[(None, H, W, in_ch), (None, num_classes)])
    img_enc = visualkeras.layered_view(
        encoder,
        legend=True,          
        draw_volume=True,     
        font=font,
        spacing=35,
        scale_xy=30,
        scale_z=0.5,
        color_map=color_map   
    )
    # img_enc.save("../result_DL/model_plot/2d_encoder_architecture.png")
    img_enc.show()


# plot dec
    decoder.build(input_shape=[(None, latent_dim), (None, num_classes)])
    img_dec = visualkeras.layered_view(
        decoder,
        legend=True,
        draw_volume=True,
        font=font,
        spacing=35,
        scale_xy=30,
        scale_z=0.5,
        color_map=color_map,
        type_ignore=[tf.keras.layers.Cropping2D]   
    )
    # img_dec.save("../result_DL/model_plot/2d_decoder_architecture.png")
    img_dec.show()



    """
    full_model.build(input_shape=[(None, H, W, in_ch), (None, num_classes)])
    img_full = visualkeras.layered_view(
        full_model,
        legend=True,
        draw_volume=True,
        font=font,
        spacing=35,
        scale_xy=30,
        scale_z=0.5,
        color_map=color_map
    )
    img_full.save("cVAE2d_full_architecture.png")
    img_full.show()
    """


c:\Users\NESS-Kuan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


##### 1D VAE plot

In [225]:
import math
import visualkeras
from PIL import ImageFont
import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    InputLayer,
    Conv1D,
    Conv1DTranspose,
    LeakyReLU,
    Flatten,
    Dense,
    Reshape,
    Concatenate,
    Cropping1D
)
from tensorflow.keras.models import Model


input_dim   = 38
latent_dim  = 10
num_classes = 2
ds_dim      = math.ceil(input_dim / 8)        # ceil(33/8) = 5
cropping_val = ds_dim * 8 - input_dim         # 5*8 - 33 = 7


color_map = {
    InputLayer:          {"fill": "#fe9775"},
    Conv1D:              {"fill": "orange"},
    LeakyReLU:           {"fill": "#ef476f"},
    Dense:               {"fill": "#118ab2"},
    Flatten:             {"fill": "#842da1"},
    Concatenate:         {"fill": "#ffbad4"},
    Reshape:             {"fill": "lightblue"}, #073b4c
    Conv1DTranspose:     {"fill": "#ffd166"},
    Cropping1D:          {"fill": "#83d483"},
    Sampling:            {"fill": "#06d6a0"},
}


class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        mu, logvar = inputs
        eps = tf.random.normal(tf.shape(mu))
        return mu + tf.exp(0.5 * logvar) * eps



# encoder
x_input = Input(shape=(input_dim,1),   name="x_input")
c_input = Input(shape=(num_classes,), name="c_input")

# Expand channel: (batch, input_dim) -> (batch, input_dim, 1)
# h = Reshape((input_dim, 1), name="expand_channel")(x_input)
h = Conv1D(64, 3, strides=2, padding="same", name="enc_conv1", data_format='channels_last')(x_input)    
h = LeakyReLU(0.2, name="enc_lrelu1")(h)
h = Conv1D(128, 3, strides=2, padding="same", name="enc_conv2", data_format='channels_last')(h)
h = LeakyReLU(0.2, name="enc_lrelu2")(h)
h = Conv1D(256, 3, strides=2, padding="same", name="enc_conv3", data_format='channels_last')(h)
h = LeakyReLU(0.2, name="enc_lrelu3")(h)

# Flatten + concat class label
h = Flatten(name="enc_flatten")(h)
h = Concatenate(name="enc_concat")([h, c_input])

h = Dense(256, name="enc_fc1")(h)
h = LeakyReLU(0.2, name="enc_lrelu4")(h)
h = Dense(128, name="enc_fc2")(h)
h = LeakyReLU(0.2, name="enc_lrelu5")(h)
h = Dense(64,  name="enc_fc3")(h)
h = LeakyReLU(0.2, name="enc_lrelu6")(h)
h = Dense(32,  name="enc_fc4")(h)
h = LeakyReLU(0.2, name="enc_lrelu7")(h)

mu     = Dense(latent_dim, name="enc_mu")(h)
logvar = Dense(latent_dim, name="enc_logvar")(h)
z      = Sampling(name="reparameterize")([mu, logvar])

encoder = Model([x_input, c_input], [z, mu, logvar], name="Encoder")




# decoder
z_input   = Input(shape=(latent_dim,),  name="z_input")
c_input_d = Input(shape=(num_classes,), name="c_input_d")

# concat z & class
d = Concatenate(name="dec_concat")([z_input, c_input_d])
d = Dense(32,  name="dec_fc1")(d)
d = LeakyReLU(0.2, name="dec_lrelu1")(d)
d = Dense(64,  name="dec_fc2")(d)
d = LeakyReLU(0.2, name="dec_lrelu2")(d)
d = Dense(128, name="dec_fc3")(d)
d = LeakyReLU(0.2, name="dec_lrelu3")(d)
d = Dense(256, name="dec_fc4")(d)
d = LeakyReLU(0.2, name="dec_lrelu4")(d)

# Fully-connected to expand for conv transpose
d = Dense(256 * ds_dim, name="dec_fc5")(d)
# reshape to (batch, length=ds_dim, channels=256)
d = Reshape((ds_dim, 256), name="dec_reshape")(d)

# 三層 Conv1DTranspose + LeakyReLU
d = Conv1DTranspose(256, 3, strides=2, padding="same", 
                    output_padding=1, name="dec_deconv1", data_format='channels_last')(d)
d = LeakyReLU(0.2, name="dec_lrelu5")(d)
d = Conv1DTranspose(128, 3, strides=2, padding="same", 
                    output_padding=1, name="dec_deconv2", data_format='channels_last')(d)
d = LeakyReLU(0.2, name="dec_lrelu6")(d)
d = Conv1DTranspose(1,  3, strides=2, padding="same", 
                    output_padding=1, name="dec_deconv3", data_format='channels_last')(d)
d = LeakyReLU(0.2, name="dec_lrelu7")(d)


# d = Conv1D(1, 3, padding="same", name="dec_conv4")(d)
d = Cropping1D(cropping=(0, cropping_val), name="crop_to_input_dim")(d)
# 移除 channel 維度 -> (batch, input_dim)
x_recon = Reshape((input_dim,), name="squeeze_channel")(d)

decoder = Model([z_input, c_input_d], x_recon, name="Decoder")




# full model
full_output = decoder([encoder.outputs[0], c_input])
cvae = Model([x_input, c_input], full_output, name="cVAE")





if __name__ == "__main__":
    try:
        font = ImageFont.truetype("../times-new-roman/times.ttf", size=40)
    except:
        font = None

    encoder.build(input_shape=[(None, input_dim), (None, num_classes)])
    enc = visualkeras.layered_view(
        encoder,
        legend=True,
        draw_volume=True,
        font=font,
        spacing=35,
        scale_xy=3,
        scale_z=0.6,
        color_map=color_map
    )
    # enc.save("../result_DL/model_plot/1d_encoder_architecture.png")
    enc.show()


    decoder.build(input_shape=[(None, latent_dim), (None, num_classes)])
    dec = visualkeras.layered_view(
        decoder,
        legend=True,
        draw_volume=True,
        font=font,
        spacing=35,
        scale_xy=3,
        scale_z=0.6,
        color_map=color_map,
        type_ignore=[tf.keras.layers.Cropping1D]
    )
    # dec.save("../result_DL/model_plot/1d_decoder_architecture.png")
    dec.show()
